# California Rice Monitoring — Macro → Micro Workflow

## Objective

This notebook does not start by looking only at 2021–2023. Instead, it follows this workflow:

```text
1. Examine long-term California rice statistics from 2000 onward
        ↓
2. Identify years with large changes in production, planted acreage, or yield
        ↓
3. Zoom in on a period where the key datasets overlap
        ↓
4. Preserve the original DWR rice-field polygons
        ↓
5. Observe USDM drought + optical + SAR signals monthly
        ↓
6. Build region- and polygon-level time series
        ↓
7. Use the observations to develop an early-supply hypothesis
```

### Analysis policy

- **NASS**: Long-term background from 2000 onward
- **DWR**: Preserve the original rice polygons without dissolving them
- **Dissolve**: Use only for the satellite query region and previews
- **USDM**: One observation per month for the detailed years, with the option to expand to weekly observations
- **Sentinel-2**: NDVI / EVI / NDMI / NDWI / MNDWI
- **Sentinel-1 SAR**: VV / VH
- **Satellite rasters**: Process primarily in Google Earth Engine
- **Google Drive cache**: Raw vectors, downloaded drought data, and feature CSV files
- Monthly GeoTIFF export can be added later if needed

### Lightning-talk context

**LT1 — Monitoring Drought Impacts on California Rice with Python**

`Long-term statistics → where/when changed → drought/water/vegetation`

**LT2 — Predicting Rice Supply with GIS and Satellite Indicators**

`Observed early-season signals → hypothesis for final yield / production`


In [ ]:
!pip -q install geopandas pyogrio shapely geemap earthengine-api mapclassify

## 0. Setup — Google Drive cache

In [ ]:
from pathlib import Path
import os
import re
import json
import zipfile
import getpass
import requests
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from shapely.geometry import mapping
from shapely import force_2d
from google.colab import drive

drive.mount("/content/drive")

# ============================================================
# USER CONFIGURATION — edit these values before running
# ============================================================

# Google Drive folder used to cache downloaded data and outputs.
# Example:
#   "/content/drive/MyDrive/california_rice_monitoring"
PROJECT_DIR = Path(
    "/content/drive/MyDrive/YOUR_PROJECT_FOLDER"
)

# Google Earth Engine Cloud Project ID.
# Example:
#   "ee-your-project-id"
EE_PROJECT = "YOUR_EARTH_ENGINE_PROJECT_ID"

if "YOUR_PROJECT_FOLDER" in str(PROJECT_DIR):
    raise ValueError(
        "Set PROJECT_DIR to your own Google Drive project folder before running."
    )

# Create the user-specified project folder if it does not already exist.
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

DATA = PROJECT_DIR / "data"
OUT = PROJECT_DIR / "output"

NASS_DIR = DATA / "nass"
DWR_DIR = DATA / "dwr"
USDM_DIR = DATA / "usdm"
SAT_DIR = DATA / "satellite_features"

for p in [DATA, OUT, NASS_DIR, DWR_DIR, USDM_DIR, SAT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# -----------------------
# Macro configuration
# -----------------------
MACRO_START_YEAR = 2000

# Rice growing season: first pass
GROWING_MONTHS = list(range(4, 11))  # Apr-Oct

AREA_CRS = "EPSG:3310"
M2_PER_ACRE = 4046.8564224

# -----------------------
# Micro configuration
# -----------------------
# Change these years after reviewing the long-term plots in Step 1.
# Current full-stack case study:
DETAIL_YEARS = [2021, 2022, 2023]

# Use one observation per month for the first pass.
OBSERVATION_MODE = "monthly"

# Polygon-level satellite processing is expensive, so start at region level.
# Set to True only after confirming the overall pattern for the lightning talk.
RUN_POLYGON_LEVEL_SATELLITE = False

# Region-level satellite first pass:
# Use 100 m rather than 30 m for the first-pass monthly regional mean to reduce processing time.
SATELLITE_REGION_SCALE = 100

# Reuse monthly cache files by default. Set to True only when recomputation is required.
FORCE_RECOMPUTE_SATELLITE = False

print("Project:", PROJECT_DIR)
print("Macro start:", MACRO_START_YEAR)
print("Detail years:", DETAIL_YEARS)

# 1. MACRO — USDA NASS California Rice Since 2000

Before looking at where drought occurred or what the satellites observed, first examine the long-term time series of:

- planted acres
- harvested acres
- yield
- production

The goal is to identify years with unusually large changes and then zoom in with detailed GIS and satellite data.


In [ ]:
NASS_RAW_CSV = (
    NASS_DIR /
    f"nass_california_rice_raw_{MACRO_START_YEAR}_latest.csv"
)

NASS_YEARLY_CSV = (
    NASS_DIR /
    f"nass_california_rice_yearly_{MACRO_START_YEAR}_latest.csv"
)

if NASS_RAW_CSV.exists():
    print("✓ NASS cache found:", NASS_RAW_CSV)
    NASS_API_KEY = None
else:
    try:
        from google.colab import userdata
        NASS_API_KEY = userdata.get("NASS_API_KEY")
    except Exception:
        NASS_API_KEY = None

    # Recommended: add a Colab secret named NASS_API_KEY.
    # If the secret is not available, the notebook will prompt for the key.
    if not NASS_API_KEY:
        NASS_API_KEY = getpass.getpass("NASS API key: ")

    assert NASS_API_KEY, "NASS API key is empty."

# Drive cache first
if NASS_RAW_CSV.exists():

    nass_raw = pd.read_csv(NASS_RAW_CSV)

else:

    url = "https://quickstats.nass.usda.gov/api/api_GET/"

    params = {
        "key": NASS_API_KEY,
        "commodity_desc": "RICE",
        "state_alpha": "CA",
        "agg_level_desc": "STATE",
        "year__GE": str(MACRO_START_YEAR),
        "format": "JSON",
    }

    r = requests.get(url, params=params, timeout=120)
    r.raise_for_status()

    nass_raw = pd.DataFrame(r.json()["data"])
    nass_raw.to_csv(NASS_RAW_CSV, index=False)

    print("✓ NASS downloaded and cached")

nass_raw["year"] = pd.to_numeric(
    nass_raw["year"], errors="coerce"
).astype("Int64")

print("rows:", len(nass_raw))
display(nass_raw.head())

def nass_value_to_numeric(s):
    return pd.to_numeric(
        s.astype(str)
         .str.replace(",", "", regex=False)
         .str.strip(),
        errors="coerce"
    )

main = nass_raw[
    nass_raw["short_desc"].str.match(r"^RICE - ", na=False)
].copy()

main["value_num"] = nass_value_to_numeric(main["Value"])

patterns = {
    "planted_acres": r"^RICE - ACRES PLANTED$",
    "harvested_acres": r"^RICE - ACRES HARVESTED$",
    "yield": r"^RICE - YIELD",
    "production": r"^RICE - PRODUCTION",
}

selected = []

for metric, pattern in patterns.items():

    tmp = main[
        main["short_desc"].str.contains(
            pattern,
            regex=True,
            na=False
        )
    ].copy()

    tmp["metric"] = metric
    selected.append(tmp)

nass_sel = pd.concat(selected, ignore_index=True)

nass_clean = nass_sel[
    (nass_sel["agg_level_desc"] == "STATE") &
    (nass_sel["freq_desc"] == "ANNUAL") &
    (nass_sel["reference_period_desc"] == "YEAR") &
    (nass_sel["source_desc"] == "SURVEY") &
    (nass_sel["domain_desc"] == "TOTAL")
].copy()

keep = (
    (
        nass_clean["metric"].isin(
            ["planted_acres", "harvested_acres"]
        )
        & (nass_clean["unit_desc"] == "ACRES")
    )
    |
    (
        (nass_clean["metric"] == "yield")
        & (nass_clean["unit_desc"] == "LB / ACRE")
    )
    |
    (
        (nass_clean["metric"] == "production")
        & (nass_clean["unit_desc"] == "CWT")
    )
)

nass_clean = nass_clean[keep].copy()

dupes = (
    nass_clean
    .groupby(["year", "metric"])
    .size()
)

bad = dupes[dupes > 1]

print("Duplicate groups:", len(bad))

if len(bad):
    display(bad)
    raise ValueError("Duplicate NASS records remain after filtering.")

nass_yearly = (
    nass_clean
    .pivot(
        index="year",
        columns="metric",
        values="value_num"
    )
    .reset_index()
)

nass_yearly["year"] = nass_yearly["year"].astype(int)
nass_yearly = nass_yearly.sort_values("year")

nass_yearly.to_csv(NASS_YEARLY_CSV, index=False)

display(nass_yearly.tail(15))

### 1-1. Inspect the long-term time series


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(
    nass_yearly["year"],
    nass_yearly["production"],
    marker="o",
    markersize=3
)
ax.set_title("California Rice Production — USDA NASS")
ax.set_xlabel("Year")
ax.set_ylabel("Production (CWT)")
ax.grid(alpha=0.3)
plt.show()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(
    nass_yearly["year"],
    nass_yearly["planted_acres"],
    marker="o",
    markersize=3,
    label="Planted acres"
)
ax.plot(
    nass_yearly["year"],
    nass_yearly["harvested_acres"],
    marker="o",
    markersize=3,
    label="Harvested acres"
)
ax.set_title("California Rice Acreage — USDA NASS")
ax.set_xlabel("Year")
ax.set_ylabel("Acres")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

## 2. MACRO → MICRO Bridge — Identify Years with Large Changes

Do not assume a cause at this stage.

First calculate year-over-year changes and identify:

> Which years are most worth investigating in detail with GIS, drought, and satellite data?


In [ ]:
change = nass_yearly.copy()

for col in [
    "production",
    "planted_acres",
    "harvested_acres",
    "yield"
]:
    change[f"{col}_yoy_pct"] = (
        change[col].pct_change() * 100
    )

# Absolute year-over-year changes
change["production_abs_change"] = (
    change["production_yoy_pct"].abs()
)

change["acreage_abs_change"] = (
    change["planted_acres_yoy_pct"].abs()
)


# =====================================================
# Production change TOP 10
# =====================================================

production_top10 = (
    change
    .sort_values(
        "production_abs_change",
        ascending=False
    )
    .head(10)
)

print("Largest production changes:")

display(
    production_top10[
        [
            "year",
            "production",
            "production_yoy_pct",
            "planted_acres_yoy_pct",
            "yield_yoy_pct"
        ]
    ]
)


# =====================================================
# Planted acreage change TOP 10
# =====================================================

acreage_top10 = (
    change
    .sort_values(
        "acreage_abs_change",
        ascending=False
    )
    .head(10)
)

print("Largest planted-acreage changes:")

display(
    acreage_top10[
        [
            "year",
            "planted_acres",
            "planted_acres_yoy_pct",
            "production_yoy_pct",
            "yield_yoy_pct"
        ]
    ]
)


# Save
change.to_csv(
    OUT / "nass_long_term_change_detection.csv",
    index=False
)

### 2-1. Select the Detailed Study Period

Review the long-term plots and the change table above, then choose the detailed study period.

Ideally:

```text
year before event → event year → year after event
```

However, prioritize a period where the following datasets are all available:

- DWR crop polygons
- Sentinel-1 / Sentinel-2
- USDM

The current first case study uses the already validated period `2021, 2022, 2023`.
Change `DETAIL_YEARS` in the Setup section if you want to rerun the workflow for different years.


In [ ]:
print("Current detailed study years:", DETAIL_YEARS)

display(
    change[
        change["year"].isin(DETAIL_YEARS)
    ][
        [
            "year",
            "planted_acres",
            "yield",
            "production",
            "planted_acres_yoy_pct",
            "yield_yoy_pct",
            "production_yoy_pct",
        ]
    ]
)

# 3. MICRO — DWR Rice Polygons

Important:

**Do not dissolve the original rice polygons.**

```text
DWR original vector
      ↓
MAIN_CROP == R1
      ↓
polygon_id + acreage
      ↓
Save to Google Drive as GeoPackage
```

Use dissolved geometry only for:

- Earth Engine query regions
- previews
- region-level summaries


In [ ]:
DWR_URLS = {
    2021: "https://data.cnra.ca.gov/dataset/6c3d65e3-35bb-49e1-a51e-49d5a2cf09a9/resource/eebd40ab-35a3-4e62-a625-0275b2849531/download/i15_crop_mapping_2021_shp.zip",
    2022: "https://data.cnra.ca.gov/dataset/6c3d65e3-35bb-49e1-a51e-49d5a2cf09a9/resource/b92e0daf-6e2e-4b5c-a112-09474138d1cd/download/i15_crop_mapping_2022_shp.zip",
    2023: "https://data.cnra.ca.gov/dataset/6c3d65e3-35bb-49e1-a51e-49d5a2cf09a9/resource/25d0f174-4bec-4987-a402-602cd1372786/download/i15_crop_mapping_final_2023.zip",
}

def download_file(url, dst):

    if dst.exists():
        print("✓ Cache:", dst.name)
        return dst

    print("Downloading:", url)

    with requests.get(
        url,
        headers={"User-Agent": "Mozilla/5.0"},
        stream=True,
        timeout=300
    ) as r:

        r.raise_for_status()

        with open(dst, "wb") as f:
            for chunk in r.iter_content(1024 * 1024):
                if chunk:
                    f.write(chunk)

    print("✓ Saved:", dst)
    return dst


def find_dwr_shp(year):

    candidates = [
        p for p in DWR_DIR.rglob("*.shp")
        if str(year) in str(p)
    ]

    if candidates:
        return max(candidates, key=lambda p: p.stat().st_size)

    return None


DWR_SHP = {}

for year in DETAIL_YEARS:

    shp = find_dwr_shp(year)

    if shp is not None:
        print(year, "✓ existing shapefile:", shp)
        DWR_SHP[year] = shp
        continue

    if year not in DWR_URLS:
        print(
            year,
            "DWR URL is not configured. "
            "Add the official URL before running this year."
        )
        continue

    zip_path = DWR_DIR / f"dwr_{year}.zip"
    download_file(DWR_URLS[year], zip_path)

    extract_dir = DWR_DIR / str(year)
    extract_dir.mkdir(parents=True, exist_ok=True)

    if not list(extract_dir.rglob("*.shp")):
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(extract_dir)

    shps = list(extract_dir.rglob("*.shp"))

    if not shps:
        raise FileNotFoundError(
            f"{year}: DWR shapefile not found after extraction."
        )

    shp = max(shps, key=lambda p: p.stat().st_size)
    DWR_SHP[year] = shp

print(DWR_SHP)

dwr = {}
rice = {}

for year, shp in DWR_SHP.items():

    print("\n====", year, "====")

    g = gpd.read_file(shp)
    dwr[year] = g

    if "MAIN_CROP" not in g.columns:
        raise KeyError(
            f"{year}: MAIN_CROP column not found."
        )

    rr = g[
        g["MAIN_CROP"].eq("R1")
    ].copy()

    rr = rr[
        rr.geometry.notna() &
        ~rr.geometry.is_empty
    ].copy()

    rr = rr.reset_index(drop=True)

    rr["year"] = year
    rr["polygon_id"] = [
        f"{year}_{i:06d}"
        for i in range(len(rr))
    ]

    rr_area = rr.to_crs(AREA_CRS)
    rr["area_acres"] = (
        rr_area.geometry.area.values /
        M2_PER_ACRE
    )

    rice[year] = rr

    # Save the original geometry without dissolving it
    gpkg = DWR_DIR / f"rice_R1_{year}.gpkg"

    if not gpkg.exists():
        rr.to_file(
            gpkg,
            layer="rice",
            driver="GPKG"
        )

    print(
        "rice polygons:",
        len(rr),
        "total acres:",
        round(rr["area_acres"].sum())
    )

### 3-1. Compare DWR GIS Acreage with NASS


In [ ]:
gis_rows = []

for year, g in rice.items():

    gis_rows.append({
        "year": year,
        "field_count": len(g),
        "gis_rice_acres": g["area_acres"].sum()
    })

gis_acres = pd.DataFrame(gis_rows)

compare_acres = (
    gis_acres
    .merge(
        nass_yearly[
            [
                "year",
                "planted_acres",
                "harvested_acres"
            ]
        ],
        on="year",
        how="left"
    )
)

compare_acres["gis_vs_nass_planted_pct"] = (
    compare_acres["gis_rice_acres"] /
    compare_acres["planted_acres"] *
    100
)

display(compare_acres)

compare_acres.to_csv(
    OUT / "dwr_vs_nass_acres.csv",
    index=False
)

# 4. MICRO — U.S. Drought Monitor

For the first pass, use **one observation per month**.

USDM is published weekly on Tuesdays, so select the Tuesday closest to the 15th of each month.

Google Drive cache first:

```text
USDM ZIP exists → reuse it
USDM ZIP does not exist → download from official URL → save to Drive
```

The same workflow can later be expanded to all Tuesdays for weekly analysis.


In [ ]:
USDM_BASE_URL = (
    "https://droughtmonitor.unl.edu/"
    "data/shapefiles_m"
)

def monthly_usdm_dates(year, months=GROWING_MONTHS):

    tuesdays = pd.date_range(
        f"{year}-01-01",
        f"{year}-12-31",
        freq="W-TUE"
    )

    chosen = []

    for month in months:

        mm = tuesdays[tuesdays.month == month]

        if len(mm) == 0:
            continue

        target = pd.Timestamp(year, month, 15)

        date = min(
            mm,
            key=lambda d: abs(d - target)
        )

        chosen.append(date)

    return chosen


def usdm_zip_path(date):

    ds = pd.Timestamp(date).strftime("%Y%m%d")

    return USDM_DIR / f"USDM_{ds}_M.zip"


def download_usdm(date):

    ds = pd.Timestamp(date).strftime("%Y%m%d")
    dst = usdm_zip_path(date)

    if dst.exists():
        print("✓ USDM cache:", dst.name)
        return dst

    url = (
        f"{USDM_BASE_URL}/"
        f"USDM_{ds}_M.zip"
    )

    return download_file(url, dst)


def read_usdm(date):

    zp = download_usdm(date)

    extract_dir = (
        USDM_DIR /
        pd.Timestamp(date).strftime("%Y%m%d")
    )

    extract_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    shps = list(
        extract_dir.rglob("*.shp")
    )

    if not shps:

        with zipfile.ZipFile(zp) as z:
            z.extractall(extract_dir)

        shps = list(
            extract_dir.rglob("*.shp")
        )

    if not shps:
        raise FileNotFoundError(
            f"No shapefile in {zp}"
        )

    shp = max(
        shps,
        key=lambda p: p.stat().st_size
    )

    g = gpd.read_file(shp)

    if "DM" not in g.columns:
        raise KeyError(
            f"DM column not found: {g.columns.tolist()}"
        )

    g["DM"] = pd.to_numeric(
        g["DM"],
        errors="coerce"
    )

    return g

# Download/cache monthly USDM data
USDM_DATES = {
    year: monthly_usdm_dates(year)
    for year in DETAIL_YEARS
}

for year, dates in USDM_DATES.items():

    print("\n====", year, "====")

    for date in dates:
        try:
            download_usdm(date)
        except Exception as e:
            print("FAILED:", date.date(), e)

### 4-1. Drought Exposure by Rice Polygon

For each rice field, calculate the intersection area with the USDM drought polygons.

Output structure:

```text
date
year
month
polygon_id
area_acres
D0_acres
D1_acres
D2_acres
D3_acres
D4_acres
D2plus_pct
```

USDM summary tables use cumulative drought categories, but this workflow overlays the `DM` category polygons from the GIS shapefiles.
Inspect the actual geometry overlaps before making a final interpretation.


In [ ]:
def drought_by_polygon(
    rice_gdf,
    drought_gdf,
    date
):

    r = (
        rice_gdf[
            [
                "polygon_id",
                "year",
                "area_acres",
                "geometry"
            ]
        ]
        .to_crs(AREA_CRS)
        .copy()
    )

    d = (
        drought_gdf[
            ["DM", "geometry"]
        ]
        .to_crs(AREA_CRS)
        .copy()
    )

    # Limit drought polygons to the vicinity of the California rice extent
    d = d[
        d.intersects(
            r.geometry.union_all()
        )
    ].copy()

    inter = gpd.overlay(
        r,
        d,
        how="intersection",
        keep_geom_type=False
    )

    inter["intersection_acres"] = (
        inter.geometry.area /
        M2_PER_ACRE
    )

    tbl = (
        inter
        .groupby(
            ["polygon_id", "DM"]
        )["intersection_acres"]
        .sum()
        .unstack(fill_value=0)
    )

    for dm in range(5):
        if dm not in tbl.columns:
            tbl[dm] = 0.0

    tbl = (
        tbl[[0, 1, 2, 3, 4]]
        .rename(
            columns={
                0: "D0_acres",
                1: "D1_acres",
                2: "D2_acres",
                3: "D3_acres",
                4: "D4_acres",
            }
        )
        .reset_index()
    )

    base = rice_gdf[
        [
            "polygon_id",
            "year",
            "area_acres"
        ]
    ].copy()

    out = base.merge(
        tbl,
        on="polygon_id",
        how="left"
    )

    drought_cols = [
        "D0_acres",
        "D1_acres",
        "D2_acres",
        "D3_acres",
        "D4_acres"
    ]

    out[drought_cols] = (
        out[drought_cols]
        .fillna(0)
    )

    out["D2plus_acres"] = (
        out["D2_acres"] +
        out["D3_acres"] +
        out["D4_acres"]
    )

    out["D2plus_pct"] = (
        out["D2plus_acres"] /
        out["area_acres"] *
        100
    )

    out["date"] = pd.Timestamp(date)
    out["month"] = pd.Timestamp(date).month

    return out

# Run:
# Reuse cached result CSVs instead of recomputing the overlay
drought_tables = []

for year in DETAIL_YEARS:

    if year not in rice:
        continue

    for date in USDM_DATES[year]:

        ds = pd.Timestamp(date).strftime("%Y%m%d")

        cache_csv = (
            USDM_DIR /
            f"rice_drought_{ds}.csv"
        )

        if cache_csv.exists():

            t = pd.read_csv(
                cache_csv,
                parse_dates=["date"]
            )

        else:

            dg = read_usdm(date)

            t = drought_by_polygon(
                rice[year],
                dg,
                date
            )

            t.to_csv(
                cache_csv,
                index=False
            )

        drought_tables.append(t)

drought_polygon_monthly = pd.concat(
    drought_tables,
    ignore_index=True
)

display(drought_polygon_monthly.head())

drought_polygon_monthly.to_csv(
    OUT / "drought_polygon_monthly.csv",
    index=False
)

# California rice acreage-weighted monthly drought summary
drought_region_monthly = (
    drought_polygon_monthly
    .groupby(["year", "month"])
    .apply(
        lambda x: pd.Series({
            "rice_acres": x["area_acres"].sum(),
            "d2plus_acres": x["D2plus_acres"].sum(),
            "d2plus_pct": (
                x["D2plus_acres"].sum() /
                x["area_acres"].sum() *
                100
            )
        }),
        include_groups=False
    )
    .reset_index()
)

display(drought_region_monthly)

### 4-2. Presentation Maps — 2021–2023

Create presentation-ready maps and charts from the existing results while minimizing recomputation.

This section does not change the underlying analysis logic.

Outputs include:
- `dwr_rice_fields_2021_2023_with_nass_production.png` — annual DWR rice-field comparison
- `rice_drought_overlay_month07_2021_2023_D3plus.png` — July USDM drought + rice-field overlay
- `d3plus_monthly_2021_2023.png` — growing-season D3+ exposure comparison

Change `MAP_MONTH = 7` to another month if needed.


In [ ]:
# ============================================================
# 4-2. Presentation maps — 2021–2023
#
# Figure 1:
#   DWR rice polygons
#   + GIS acreage
#   + NASS planted / harvested acreage
#   + production
#   + average yield
#
# Figure 2:
#   USDM drought background
#   + rice polygon boundaries
#   + D3+ (Extreme-or-worse) rice exposure
#
# Figure 3:
#   Monthly D3+ drought exposure
#
# Existing analysis results are reused.
# No satellite recomputation.
# ============================================================

from shapely.geometry import box
from matplotlib.patches import Patch


# ------------------------------------------------------------
# Output folder
# ------------------------------------------------------------

PRESENTATION_DIR = OUT / "presentation_maps"
PRESENTATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# Years actually available
# ------------------------------------------------------------

MAP_YEARS = [
    y for y in DETAIL_YEARS
    if y in rice
]

if not MAP_YEARS:
    raise RuntimeError(
        "No rice polygons are loaded. "
        "Run the DWR section first."
    )


# ------------------------------------------------------------
# Project rice polygons once
# ------------------------------------------------------------

rice_plot = {
    year: rice[year]
        .to_crs(AREA_CRS)
        .copy()
    for year in MAP_YEARS
}


# ------------------------------------------------------------
# Common map extent across all detail years
# ------------------------------------------------------------

all_bounds = np.vstack([
    g.total_bounds
    for g in rice_plot.values()
])

xmin = all_bounds[:, 0].min()
ymin = all_bounds[:, 1].min()
xmax = all_bounds[:, 2].max()
ymax = all_bounds[:, 3].max()

pad_x = (xmax - xmin) * 0.04
pad_y = (ymax - ymin) * 0.04

xmin -= pad_x
xmax += pad_x
ymin -= pad_y
ymax += pad_y

view_box = box(
    xmin,
    ymin,
    xmax,
    ymax
)


# ============================================================
# Presentation metric:
# D3+ = Extreme Drought or worse
#     = D3 + D4
#
# Keep the existing D2+ analysis unchanged.
# This is only for presentation.
# ============================================================

d3tmp = drought_polygon_monthly.copy()

d3tmp["D3plus_acres"] = (
    d3tmp["D3_acres"] +
    d3tmp["D4_acres"]
)

d3plus_region_monthly = (
    d3tmp
    .groupby(
        ["year", "month"],
        as_index=False
    )
    .agg(
        rice_acres=(
            "area_acres",
            "sum"
        ),
        d3plus_acres=(
            "D3plus_acres",
            "sum"
        )
    )
)

d3plus_region_monthly["d3plus_pct"] = (
    d3plus_region_monthly[
        "d3plus_acres"
    ]
    /
    d3plus_region_monthly[
        "rice_acres"
    ]
    * 100
)

print(
    "D3+ presentation summary "
    "(Extreme or Exceptional Drought)"
)

display(
    d3plus_region_monthly
)


# ============================================================
# Figure 1
# California rice fields
# + acreage
# + production
# + yield
# ============================================================

fig, axes = plt.subplots(
    1,
    len(MAP_YEARS),
    figsize=(18, 6),
    constrained_layout=True
)

if len(MAP_YEARS) == 1:
    axes = [axes]


for ax, year in zip(
    axes,
    MAP_YEARS
):

    g = rice_plot[year]

    # --------------------------------------------------------
    # Rice polygons
    # --------------------------------------------------------

    g.plot(
        ax=ax,
        color="#5B8DB8",
        edgecolor="black",
        linewidth=0.12
    )

    gis_acres = (
        g["area_acres"]
        .sum()
    )

    # --------------------------------------------------------
    # Acreage from compare_acres
    # --------------------------------------------------------

    acreage_row = (
        compare_acres[
            compare_acres["year"] == year
        ]
        .iloc[0]
    )

    planted = (
        float(
            acreage_row[
                "planted_acres"
            ]
        )
        if pd.notna(
            acreage_row[
                "planted_acres"
            ]
        )
        else np.nan
    )

    harvested = (
        float(
            acreage_row[
                "harvested_acres"
            ]
        )
        if pd.notna(
            acreage_row[
                "harvested_acres"
            ]
        )
        else np.nan
    )

    # --------------------------------------------------------
    # Production / yield directly from nass_yearly
    # --------------------------------------------------------

    nass_row = (
        nass_yearly[
            nass_yearly["year"] == year
        ]
        .iloc[0]
    )

    production_cwt = float(
        nass_row["production"]
    )

    yield_lb_ac = float(
        nass_row["yield"]
    )

    production_txt = (
        f"{production_cwt / 1_000_000:.1f}M CWT"
    )

    yield_txt = (
        f"{yield_lb_ac:,.0f} lb/ac"
    )

    # Same yield expressed in CWT / harvested acre
    if (
        pd.notna(harvested)
        and harvested > 0
    ):
        yield_cwt_ac = (
            production_cwt /
            harvested
        )
        yield_cwt_txt = (
            f"{yield_cwt_ac:.1f} CWT/ac"
        )
    else:
        yield_cwt_txt = "n/a"

    # --------------------------------------------------------
    # Title for each year
    # --------------------------------------------------------

    ax.set_title(
        f"{year}\n"
        f"{len(g):,} rice polygons | "
        f"GIS acres: {gis_acres:,.0f}\n"
        f"NASS planted: {planted:,.0f} | "
        f"harvested: {harvested:,.0f}\n"
        f"Production: {production_txt} | "
        f"Yield: {yield_txt} "
        f"({yield_cwt_txt})",
        fontsize=10
    )

    ax.set_xlim(
        xmin,
        xmax
    )

    ax.set_ylim(
        ymin,
        ymax
    )

    ax.set_axis_off()


fig.suptitle(
    "California Rice Fields — DWR Crop Mapping",
    fontsize=16
)

rice_map_path = (
    PRESENTATION_DIR /
    "dwr_rice_fields_2021_2023_with_nass_production.png"
)

fig.savefig(
    rice_map_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(
    "✓ Saved:",
    rice_map_path
)


# ============================================================
# Figure 2
# USDM drought background
# + rice field boundaries
#
# Presentation metric:
# D3+ = Extreme or Exceptional Drought
# ============================================================

MAP_MONTH = 7


DM_LABELS = {
    0: "D0 — Abnormally Dry",
    1: "D1 — Moderate Drought",
    2: "D2 — Severe Drought",
    3: "D3 — Extreme Drought",
    4: "D4 — Exceptional Drought",
}


DM_COLORS = {
    0: "#FFFF00",
    1: "#FCD37F",
    2: "#FFAA00",
    3: "#E60000",
    4: "#730000",
}


# Keep map and legend at the SAME alpha
DROUGHT_ALPHA = 0.35


fig, axes = plt.subplots(
    1,
    len(MAP_YEARS),
    figsize=(16, 6),
    constrained_layout=True
)

if len(MAP_YEARS) == 1:
    axes = [axes]


for ax, year in zip(
    axes,
    MAP_YEARS
):

    # --------------------------------------------------------
    # Select the monthly USDM observation
    # --------------------------------------------------------

    dates = [
        d
        for d in USDM_DATES[year]
        if (
            pd.Timestamp(d).month
            == MAP_MONTH
        )
    ]

    if not dates:

        ax.set_title(
            f"{year}\n"
            f"No USDM date for month "
            f"{MAP_MONTH}"
        )

        ax.set_axis_off()
        continue


    date = dates[0]


    # --------------------------------------------------------
    # Read USDM
    # --------------------------------------------------------

    drought_gdf = (
        read_usdm(date)
        .to_crs(AREA_CRS)
        .copy()
    )


    # --------------------------------------------------------
    # Keep polygons intersecting the visible map extent.
    #
    # IMPORTANT:
    # This is NOT clipping by rice fields.
    # It only avoids plotting drought polygons far outside
    # the common display area.
    # --------------------------------------------------------

    drought_gdf = drought_gdf[
        drought_gdf.intersects(
            view_box
        )
    ].copy()


    # --------------------------------------------------------
    # Draw USDM drought background FIRST
    # --------------------------------------------------------

    if not drought_gdf.empty:

        for dm in sorted(
            DM_COLORS.keys()
        ):

            part = drought_gdf[
                drought_gdf["DM"]
                == dm
            ]

            if not part.empty:

                part.plot(
                    ax=ax,
                    color=DM_COLORS[dm],
                    alpha=DROUGHT_ALPHA,
                    linewidth=0
                )


    # --------------------------------------------------------
    # Draw rice field boundaries ON TOP
    # --------------------------------------------------------

    rice_plot[
        year
    ].boundary.plot(
        ax=ax,
        color="black",
        linewidth=0.28
    )


    # --------------------------------------------------------
    # D3+ exposure:
    # D3 Extreme + D4 Exceptional
    # --------------------------------------------------------

    d3_row = (
        d3plus_region_monthly[
            (
                d3plus_region_monthly[
                    "year"
                ]
                == year
            )
            &
            (
                d3plus_region_monthly[
                    "month"
                ]
                == MAP_MONTH
            )
        ]
    )


    if len(d3_row):

        d3_pct = float(
            d3_row.iloc[0][
                "d3plus_pct"
            ]
        )

        d3_txt = (
            "D3+ rice exposure: "
            f"{d3_pct:.1f}%"
        )

    else:

        d3_txt = (
            "D3+ rice exposure: n/a"
        )


    # --------------------------------------------------------
    # Map title
    # --------------------------------------------------------

    ax.set_title(
        f"{year} — "
        f"{pd.Timestamp(date).date()}\n"
        f"{d3_txt}",
        fontsize=11
    )


    ax.set_xlim(
        xmin,
        xmax
    )

    ax.set_ylim(
        ymin,
        ymax
    )

    ax.set_axis_off()


# ------------------------------------------------------------
# Shared USDM legend
#
# Same colors AND same alpha as map.
# Rice polygons are intentionally NOT included in legend.
# ------------------------------------------------------------

legend_items = []

for dm in sorted(
    DM_COLORS.keys()
):

    legend_items.append(
        Patch(
            facecolor=DM_COLORS[dm],
            edgecolor="none",
            alpha=DROUGHT_ALPHA,
            label=DM_LABELS[dm]
        )
    )


fig.legend(
    handles=legend_items,
    loc="lower center",
    ncol=5,
    bbox_to_anchor=(
        0.5,
        -0.05
    ),
    frameon=False
)


fig.suptitle(
    "Rice Fields and U.S. Drought Monitor "
    f"— Month {MAP_MONTH:02d}",
    fontsize=16
)


drought_map_path = (
    PRESENTATION_DIR /
    f"rice_drought_overlay_month{MAP_MONTH:02d}"
    "_2021_2023_D3plus.png"
)


fig.savefig(
    drought_map_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(
    "✓ Saved:",
    drought_map_path
)


# ============================================================
# Figure 3
# Monthly share of rice acreage under D3+ drought
#
# D3+ = Extreme or Exceptional Drought
# ============================================================

fig, ax = plt.subplots(
    figsize=(10, 5)
)


for year in MAP_YEARS:

    g = (
        d3plus_region_monthly[
            d3plus_region_monthly[
                "year"
            ]
            == year
        ]
        .sort_values(
            "month"
        )
    )

    ax.plot(
        g["month"],
        g["d3plus_pct"],
        marker="o",
        label=str(year)
    )


ax.set_xticks(
    GROWING_MONTHS
)

ax.set_xlabel(
    "Month"
)

ax.set_ylabel(
    "Rice acreage under D3+ drought (%)"
)

ax.set_title(
    "Monthly Extreme-or-Worse Drought Exposure "
    "in California Rice Areas"
)

ax.grid(
    alpha=0.3
)

ax.legend(
    title="Year"
)


d3_chart_path = (
    PRESENTATION_DIR /
    "d3plus_monthly_2021_2023.png"
)


fig.savefig(
    d3_chart_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(
    "✓ Saved:",
    d3_chart_path
)


# ============================================================
# Compact presentation table
# Acreage + Production + Yield
# ============================================================

presentation_acres = (
    compare_acres[
        [
            "year",
            "field_count",
            "gis_rice_acres",
            "planted_acres",
            "harvested_acres",
            "gis_vs_nass_planted_pct"
        ]
    ]
    .merge(
        nass_yearly[
            [
                "year",
                "production",
                "yield"
            ]
        ],
        on="year",
        how="left"
    )
    .merge(
        d3plus_region_monthly[
            d3plus_region_monthly[
                "month"
            ]
            == MAP_MONTH
        ][
            [
                "year",
                "d3plus_pct"
            ]
        ],
        on="year",
        how="left"
    )
    .copy()
)


# ------------------------------------------------------------
# Derived values
# ------------------------------------------------------------

presentation_acres[
    "production_million_cwt"
] = (
    presentation_acres[
        "production"
    ]
    / 1_000_000
)


presentation_acres[
    "yield_cwt_per_acre"
] = (
    presentation_acres[
        "production"
    ]
    /
    presentation_acres[
        "harvested_acres"
    ]
)


# ------------------------------------------------------------
# Formatting
# ------------------------------------------------------------

presentation_acres[
    "gis_rice_acres"
] = (
    presentation_acres[
        "gis_rice_acres"
    ]
    .round(0)
    .astype("Int64")
)


presentation_acres[
    "planted_acres"
] = (
    presentation_acres[
        "planted_acres"
    ]
    .round(0)
    .astype("Int64")
)


presentation_acres[
    "harvested_acres"
] = (
    presentation_acres[
        "harvested_acres"
    ]
    .round(0)
    .astype("Int64")
)


presentation_acres[
    "production_million_cwt"
] = (
    presentation_acres[
        "production_million_cwt"
    ]
    .round(1)
)


presentation_acres[
    "yield"
] = (
    presentation_acres[
        "yield"
    ]
    .round(0)
)


presentation_acres[
    "yield_cwt_per_acre"
] = (
    presentation_acres[
        "yield_cwt_per_acre"
    ]
    .round(1)
)


presentation_acres[
    "gis_vs_nass_planted_pct"
] = (
    presentation_acres[
        "gis_vs_nass_planted_pct"
    ]
    .round(1)
)


presentation_acres[
    "d3plus_pct"
] = (
    presentation_acres[
        "d3plus_pct"
    ]
    .round(1)
)


# ------------------------------------------------------------
# Display compact table
# ------------------------------------------------------------

display(
    presentation_acres[
        [
            "year",
            "field_count",
            "gis_rice_acres",
            "planted_acres",
            "harvested_acres",
            "production_million_cwt",
            "yield",
            "yield_cwt_per_acre",
            "d3plus_pct",
            "gis_vs_nass_planted_pct"
        ]
    ]
)


# ------------------------------------------------------------
# Save table
# ------------------------------------------------------------

presentation_acres.to_csv(
    PRESENTATION_DIR /
    "acreage_summary_2021_2023_with_D3plus.csv",
    index=False
)


print(
    "✓ Presentation outputs:",
    PRESENTATION_DIR
)

# 5. MICRO — Monthly Satellite Observations

## Optical
Sentinel-2:
- NDVI
- EVI
- NDMI
- NDWI
- MNDWI

## SAR
Sentinel-1:
- VV
- VH
- VV − VH

The first pass uses **monthly median composites**.

### Geometry policy

- Original DWR polygons: preserve them unchanged
- Dissolved geometry: use only as the query region
- After satellite retrieval:
  - first pass → region mean
  - next step → `reduceRegions` over the original polygons


In [ ]:
import ee
import geemap

if EE_PROJECT == "YOUR_EARTH_ENGINE_PROJECT_ID":
    raise ValueError(
        "Set EE_PROJECT to your Google Earth Engine Cloud Project ID in the Setup section before running."
    )

try:
    ee.Initialize(project=EE_PROJECT)

except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)

print("EE initialized:", EE_PROJECT)

def rice_region_to_ee(
    rice_gdf,
    simplify_m=500
):
    # Do not modify the original rice_gdf
    g = rice_gdf.to_crs(
        AREA_CRS
    ).copy()

    geom = g.geometry.union_all()
    geom = force_2d(geom)

    geom = geom.simplify(
        simplify_m,
        preserve_topology=True
    )

    geom_wgs84 = (
        gpd.GeoSeries(
            [geom],
            crs=AREA_CRS
        )
        .to_crs("EPSG:4326")
        .iloc[0]
    )

    gj = mapping(geom_wgs84)

    if gj["type"] == "MultiPolygon":

        return ee.Geometry.MultiPolygon(
            gj["coordinates"],
            proj="EPSG:4326",
            geodesic=False
        )

    if gj["type"] == "Polygon":

        return ee.Geometry.Polygon(
            gj["coordinates"],
            proj="EPSG:4326",
            geodesic=False
        )

    raise ValueError(gj["type"])


ee_rice_regions = {
    year: rice_region_to_ee(rice[year])
    for year in DETAIL_YEARS
    if year in rice
}

S2_SR = ee.ImageCollection(
    "COPERNICUS/S2_SR_HARMONIZED"
)

S2_CLOUD = ee.ImageCollection(
    "COPERNICUS/S2_CLOUD_PROBABILITY"
)

S1 = ee.ImageCollection(
    "COPERNICUS/S1_GRD"
)

MAX_CLOUD_PROB = 40


def s2_mask_edges(img):
    return img.updateMask(
        img.select("B8A").mask()
        .updateMask(
            img.select("B9").mask()
        )
    )


def s2_mask_clouds(img):
    cloud = ee.Image(
        img.get("cloud_mask")
    ).select("probability")

    return img.updateMask(
        cloud.lt(MAX_CLOUD_PROB)
    )


def add_optical_indices(img):

    ndvi = (
        img.normalizedDifference(
            ["B8", "B4"]
        )
        .rename("NDVI")
    )

    ndmi = (
        img.normalizedDifference(
            ["B8", "B11"]
        )
        .rename("NDMI")
    )

    ndwi = (
        img.normalizedDifference(
            ["B3", "B8"]
        )
        .rename("NDWI")
    )

    mndwi = (
        img.normalizedDifference(
            ["B3", "B11"]
        )
        .rename("MNDWI")
    )

    refl = (
        img.select(
            ["B2", "B4", "B8"]
        )
        .multiply(0.0001)
    )

    blue = refl.select("B2")
    red = refl.select("B4")
    nir = refl.select("B8")

    evi = (
        nir.subtract(red)
        .multiply(2.5)
        .divide(
            nir
            .add(red.multiply(6))
            .subtract(blue.multiply(7.5))
            .add(1)
        )
        .rename("EVI")
    )

    return img.addBands(
        [ndvi, evi, ndmi, ndwi, mndwi]
    )


def get_s2(region, start, end):

    criteria = ee.Filter.And(
        ee.Filter.bounds(region),
        ee.Filter.date(start, end)
    )

    sr = (
        S2_SR
        .filter(criteria)
        .map(s2_mask_edges)
    )

    clouds = S2_CLOUD.filter(criteria)

    joined = ee.Join.saveFirst(
        "cloud_mask"
    ).apply(
        primary=sr,
        secondary=clouds,
        condition=ee.Filter.equals(
            leftField="system:index",
            rightField="system:index"
        )
    )

    return (
        ee.ImageCollection(joined)
        .map(s2_mask_clouds)
        .map(add_optical_indices)
    )


def get_s1(region, start, end):

    return (
        S1
        .filterBounds(region)
        .filterDate(start, end)
        .filter(
            ee.Filter.eq(
                "instrumentMode",
                "IW"
            )
        )
        .filter(
            ee.Filter.listContains(
                "transmitterReceiverPolarisation",
                "VV"
            )
        )
        .filter(
            ee.Filter.listContains(
                "transmitterReceiverPolarisation",
                "VH"
            )
        )
        .select(["VV", "VH"])
    )

### 5-1. Region-level monthly satellite time series

In [ ]:
def monthly_region_satellite(
    year,
    region,
    months=GROWING_MONTHS,
    scale=SATELLITE_REGION_SCALE
):
    """
    Region-level monthly satellite summary.

    Performance notes:
    - Retrieve S2/S1 image counts with one ee.Dictionary(...).getInfo() call
    - Combine optical + SAR into one multi-band image,
      then run reduceRegion().getInfo() only once per month
    - Use scale=100 m for the first pass

    This reduces synchronous Earth Engine calls from up to four per month to about two per month.
    """

    rows = []

    for month in months:

        start = ee.Date.fromYMD(year, month, 1)
        end = start.advance(1, "month")

        s2 = get_s2(region, start, end)
        s1 = get_s1(region, start, end)

        # First synchronous call: retrieve S2/S1 counts together
        counts = ee.Dictionary({
            "s2_images": s2.size(),
            "s1_images": s1.size(),
        }).getInfo()

        n_s2 = int(counts["s2_images"])
        n_s1 = int(counts["s1_images"])

        row = {
            "year": year,
            "month": month,
            "s2_images": n_s2,
            "s1_images": n_s1,
            "ndvi": None,
            "evi": None,
            "ndmi": None,
            "ndwi": None,
            "mndwi": None,
            "sar_vv": None,
            "sar_vh": None,
            "sar_vv_minus_vh": None,
        }

        images_to_combine = []

        if n_s2 > 0:
            optical = (
                s2
                .select(["NDVI", "EVI", "NDMI", "NDWI", "MNDWI"])
                .median()
            )
            images_to_combine.append(optical)

        if n_s1 > 0:
            sar = s1.median()
            vv_minus_vh = (
                sar.select("VV")
                .subtract(sar.select("VH"))
                .rename("VV_minus_VH")
            )
            sar_bands = (
                sar.select(["VV", "VH"])
                .addBands(vv_minus_vh)
            )
            images_to_combine.append(sar_bands)

        # Second synchronous call: combine optical/SAR and run reduceRegion once
        if images_to_combine:
            combined = images_to_combine[0]
            for extra_img in images_to_combine[1:]:
                combined = combined.addBands(extra_img)

            stats = combined.reduceRegion(
                reducer=ee.Reducer.mean(),
                geometry=region,
                scale=scale,
                bestEffort=True,
                maxPixels=1e9,
                tileScale=4
            ).getInfo()

            row.update({
                "ndvi": stats.get("NDVI"),
                "evi": stats.get("EVI"),
                "ndmi": stats.get("NDMI"),
                "ndwi": stats.get("NDWI"),
                "mndwi": stats.get("MNDWI"),
                "sar_vv": stats.get("VV"),
                "sar_vh": stats.get("VH"),
                "sar_vv_minus_vh": stats.get("VV_minus_VH"),
            })

        rows.append(row)

    return pd.DataFrame(rows)

### 5-1a. Optional Smoke Test — One Month Only

Before running all 3 years × 7 months, you can test only July 2022.

The test cell below is commented out. Uncomment it only when you want to run the smoke test.

Use `SATELLITE_REGION_SCALE = 100` for the first pass. If the result looks reasonable, continue with the full monthly run.


In [ ]:
# test_sat = monthly_region_satellite(
#     2022,
#     ee_rice_regions[2022],
#     months=[7],
#     scale=SATELLITE_REGION_SCALE
# )
# display(test_sat)

In [ ]:
# ============================================================
# Satellite snapshot cache builder
# - One representative month only
# - If cached PNG exists, do NOT download again
# - If not, download from GEE and save to Drive
#
# Columns later: 2021, 2022, 2023
# Rows later: selected satellite / indicator types
# ============================================================

from pathlib import Path
from io import BytesIO
from PIL import Image
import requests
import calendar

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

SNAP_MONTH = 8   # August; change to 7 for July if you prefer

SNAP_YEARS = [
    y for y in [2021, 2022, 2023]
    if y in ee_rice_regions
]

# Rows to display later
SNAP_ROWS = [
    ("rgb",    "True Color"),
    ("ndvi",   "NDVI"),
    ("ndmi",   "NDMI"),
    ("mndwi",  "MNDWI"),
    ("sar_vh", "SAR VH"),
]

THUMB_DIMENSION = 700

SNAP_DIR = OUT / "presentation_maps" / f"satellite_snapshot_month{SNAP_MONTH:02d}"
SNAP_DIR.mkdir(parents=True, exist_ok=True)

print("Snapshot folder:", SNAP_DIR)
print("Month:", SNAP_MONTH, calendar.month_name[SNAP_MONTH])
print("Years:", SNAP_YEARS)


# ------------------------------------------------------------
# Fixed visualization palettes / scales
# Keep scales fixed across years for fair comparison
# ------------------------------------------------------------

NDVI_PALETTE = [
    "7f3b08", "b35806", "f1a340", "fee08b",
    "d9ef8b", "66bd63", "1a9850", "006837"
]

MOISTURE_PALETTE = [
    "8c510a", "d8b365", "f6e8c3",
    "c7eae5", "5ab4ac", "01665e"
]

WATER_PALETTE = [
    "8c510a", "f6e8c3", "d9f0f0",
    "67a9cf", "2166ac"
]

SAR_PALETTE = [
    "000000", "FFFFFF"
]


# ------------------------------------------------------------
# Helper: build monthly composites in GEE
# ------------------------------------------------------------

def build_monthly_snapshot(year, month, region):
    start = ee.Date.fromYMD(year, month, 1)
    end = start.advance(1, "month")

    s2 = get_s2(region, start, end)
    s1 = get_s1(region, start, end)

    counts = ee.Dictionary({
        "s2": s2.size(),
        "s1": s1.size(),
    }).getInfo()

    n_s2 = int(counts["s2"])
    n_s1 = int(counts["s1"])

    optical = None
    sar = None

    if n_s2 > 0:
        optical = s2.median()

    if n_s1 > 0:
        sar = s1.median()

    return {
        "optical": optical,
        "sar": sar,
        "n_s2": n_s2,
        "n_s1": n_s1,
    }


# ------------------------------------------------------------
# Helper: visualization image for each indicator
# ------------------------------------------------------------

def get_vis_image(key, optical, sar):

    # Sentinel-2
    if key == "rgb":
        if optical is None:
            return None
        return optical.select(["B4", "B3", "B2"]).visualize(
            min=0, max=3000, gamma=1.2
        )

    if key == "ndvi":
        if optical is None:
            return None
        return optical.select("NDVI").visualize(
            min=-0.2, max=0.9, palette=NDVI_PALETTE
        )

    if key == "ndmi":
        if optical is None:
            return None
        return optical.select("NDMI").visualize(
            min=-0.5, max=0.7, palette=MOISTURE_PALETTE
        )

    if key == "mndwi":
        if optical is None:
            return None
        return optical.select("MNDWI").visualize(
            min=-0.7, max=0.6, palette=WATER_PALETTE
        )

    # Sentinel-1
    if key == "sar_vh":
        if sar is None:
            return None
        return sar.select("VH").visualize(
            min=-35, max=-5, palette=SAR_PALETTE
        )

    raise ValueError(f"Unknown key: {key}")


# ------------------------------------------------------------
# Helper: save GEE thumbnail only if not cached
# ------------------------------------------------------------

def save_thumbnail_if_needed(vis_image, region, save_path, dimension=THUMB_DIMENSION):

    if save_path.exists():
        print(f"  Cached: {save_path.name}")
        return

    if vis_image is None:
        print(f"  No data: {save_path.name}")
        return

    params = {
        "region": region.bounds(),
        "dimensions": dimension,
        "format": "png",
    }

    url = vis_image.getThumbURL(params)

    response = requests.get(url, timeout=180)
    response.raise_for_status()

    img = Image.open(BytesIO(response.content)).convert("RGBA")

    # make transparent background white
    background = Image.new("RGBA", img.size, "white")
    background.alpha_composite(img)
    img_rgb = background.convert("RGB")

    img_rgb.save(save_path)
    print(f"  Saved: {save_path.name}")


# ------------------------------------------------------------
# Main loop: cache images
# ------------------------------------------------------------

for year in SNAP_YEARS:
    print(f"\n===== Preparing {year}-{SNAP_MONTH:02d} =====")

    region = ee_rice_regions[year]

    data = build_monthly_snapshot(year, SNAP_MONTH, region)

    print(f"  Sentinel-2 scenes: {data['n_s2']}")
    print(f"  Sentinel-1 scenes: {data['n_s1']}")

    for key, label in SNAP_ROWS:
        save_path = SNAP_DIR / f"{year}_{SNAP_MONTH:02d}_{key}.png"
        vis_image = get_vis_image(key, data["optical"], data["sar"])
        save_thumbnail_if_needed(vis_image, region, save_path)

print("\nDone.")
print("Saved / cached PNGs in:", SNAP_DIR)

In [ ]:
# ============================================================
# Display cached satellite snapshot panel
# Columns = 2021, 2022, 2023
# Rows    = selected satellite / indicator types
# ============================================================

from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import calendar

# ============================================================
# Compact satellite panel for PowerPoint
#
# Columns = 2021 / 2022 / 2023
# Rows    = satellite / indicator types
# ============================================================

n_rows = len(SNAP_ROWS)
n_cols = len(SNAP_YEARS)

# Compact landscape-ish figure
fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(8.2, 7.2)
)

# make sure axes is always 2D
if n_rows == 1 and n_cols == 1:
    axes = np.array([[axes]])
elif n_rows == 1:
    axes = np.array([axes])
elif n_cols == 1:
    axes = np.array([[ax] for ax in axes])


for col_idx, year in enumerate(SNAP_YEARS):

    for row_idx, (key, row_label) in enumerate(SNAP_ROWS):

        ax = axes[row_idx, col_idx]

        img_path = (
            SNAP_DIR /
            f"{year}_{SNAP_MONTH:02d}_{key}.png"
        )

        if img_path.exists():

            img = (
                Image.open(img_path)
                .convert("RGB")
            )

            ax.imshow(img)

        else:

            ax.text(
                0.5,
                0.5,
                "Missing",
                ha="center",
                va="center",
                transform=ax.transAxes,
                fontsize=8
            )

            ax.set_facecolor("#EEEEEE")

        ax.set_axis_off()

        # --------------------------------------------
        # Year only at top
        # --------------------------------------------
        if row_idx == 0:

            ax.set_title(
                str(year),
                fontsize=11,
                fontweight="bold",
                pad=2
            )

        # --------------------------------------------
        # Compact row labels
        # Horizontal labels are easier to read
        # --------------------------------------------
        if col_idx == 0:

            ax.text(
                -0.03,
                0.5,
                row_label,
                transform=ax.transAxes,
                rotation=90,
                va="center",
                ha="right",
                fontsize=9,
                fontweight="bold"
            )


# ============================================================
# Very compact spacing
# ============================================================

plt.subplots_adjust(
    left=0.075,
    right=0.995,
    top=0.95,
    bottom=0.015,
    wspace=0.015,
    hspace=0.025
)


# ============================================================
# Save compact PowerPoint-ready PNG
# ============================================================

compact_panel_path = (
    SNAP_DIR /
    f"satellite_panel_compact_{SNAP_MONTH:02d}_2021_2023.png"
)

fig.savefig(
    compact_panel_path,
    dpi=300,
    bbox_inches="tight",
    pad_inches=0.02,
    facecolor="white"
)

plt.show()

print("✓ Saved:", compact_panel_path)

In [ ]:
sat_region_tables = []

for year, region in ee_rice_regions.items():

    print(f"\n========== {year} ==========")

    year_tables = []

    for month in GROWING_MONTHS:

        # Monthly cache:
        # Completed months are reused if the Colab session stops and is restarted.
        cache_csv = (
            SAT_DIR /
            f"satellite_region_{year}_{month:02d}.csv"
        )

        if (
            cache_csv.exists()
            and not FORCE_RECOMPUTE_SATELLITE
        ):
            print(f"{year}-{month:02d}: ✓ cache")
            t = pd.read_csv(cache_csv)

        else:
            print(
                f"{year}-{month:02d}: "
                f"calculating at {SATELLITE_REGION_SCALE} m ..."
            )

            t = monthly_region_satellite(
                year,
                region,
                months=[month],
                scale=SATELLITE_REGION_SCALE
            )

            t.to_csv(cache_csv, index=False)
            print(f"{year}-{month:02d}: ✓ saved")

        year_tables.append(t)

    year_table = pd.concat(
        year_tables,
        ignore_index=True
    )
    sat_region_tables.append(year_table)

sat_region_monthly = pd.concat(
    sat_region_tables,
    ignore_index=True
)

sat_region_monthly = (
    sat_region_monthly
    .sort_values(["year", "month"])
    .reset_index(drop=True)
)

display(sat_region_monthly)

sat_region_monthly.to_csv(
    OUT / "satellite_region_monthly.csv",
    index=False
)


# ============================================================
# Satellite Indicators — all monthly time-series plots
#
# Uses existing sat_region_monthly table.
# NO Earth Engine recomputation.
#
# 4 rows x 2 columns:
# NDVI / EVI / NDMI / NDWI
# MNDWI / SAR VV / SAR VH / VV-VH
# ============================================================

import matplotlib.pyplot as plt
import calendar

# ------------------------------------------------------------
# Indicators to plot
# ------------------------------------------------------------

INDICATORS = [
    ("ndvi", "NDVI", "Mean NDVI"),
    ("evi", "EVI", "Mean EVI"),
    ("ndmi", "NDMI", "Mean NDMI"),
    ("ndwi", "NDWI", "Mean NDWI"),
    ("mndwi", "MNDWI", "Mean MNDWI"),
    ("sar_vv", "SAR VV", "Mean VV backscatter (dB)"),
    ("sar_vh", "SAR VH", "Mean VH backscatter (dB)"),
    (
        "sar_vv_minus_vh",
        "SAR VV − VH",
        "Mean VV − VH (dB)"
    ),
]

# Month labels
month_labels = [
    calendar.month_abbr[m]
    for m in GROWING_MONTHS
]

# ------------------------------------------------------------
# Create panel
# ------------------------------------------------------------

fig, axes = plt.subplots(
    4,
    2,
    figsize=(12, 14)
)

axes = axes.flatten()

for ax, (column, title, ylabel) in zip(
    axes,
    INDICATORS
):

    for year, g in (
        sat_region_monthly
        .groupby("year")
    ):

        g = g.sort_values("month")

        ax.plot(
            g["month"],
            g[column],
            marker="o",
            linewidth=2,
            label=str(year)
        )

    ax.set_title(
        title,
        fontsize=12,
        fontweight="bold"
    )

    ax.set_ylabel(
        ylabel
    )

    ax.set_xticks(
        GROWING_MONTHS
    )

    ax.set_xticklabels(
        month_labels
    )

    ax.grid(
        alpha=0.3
    )

    ax.legend(
        title="Year",
        fontsize=8
    )


fig.suptitle(
    "Monthly Satellite Indicators over California Rice Areas",
    fontsize=17,
    fontweight="bold"
)

plt.tight_layout(
    rect=[0, 0, 1, 0.97]
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

all_indicator_path = (
    OUT /
    "presentation_maps" /
    "satellite_indicators_all_2021_2023.png"
)

fig.savefig(
    all_indicator_path,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

print(
    "✓ Saved:",
    all_indicator_path
)

### 5-2. Optional — Return Satellite Metrics to the Original Polygons

This is the next micro-scale analysis step.

**Preserve the original polygons, build the monthly composite, and then calculate zonal statistics for each polygon.**

Because this may involve several thousand to more than ten thousand fields, the workflow uses batches.

Keep `RUN_POLYGON_LEVEL_SATELLITE = False` while checking the regional pattern for the lightning talk.
Change it to `True` only when polygon-level processing is needed.


In [ ]:
def gdf_chunk_to_ee(
    gdf,
    simplify_m=0
):

    g = gdf.to_crs(
        "EPSG:4326"
    ).copy()

    features = []

    for _, row in g.iterrows():

        geom = force_2d(
            row.geometry
        )

        if simplify_m > 0:
            # Do not simplify in meters while the geometry is in WGS84.
            # If needed, simplify after projecting to a metric CRS.
            pass

        gj = json.loads(
            json.dumps(
                mapping(geom)
            )
        )

        feat = ee.Feature(
            ee.Geometry(gj),
            {
                "polygon_id": row["polygon_id"],
                "year": int(row["year"]),
                "area_acres": float(row["area_acres"])
            }
        )

        features.append(feat)

    return ee.FeatureCollection(features)


def polygon_month_satellite(
    rice_gdf,
    year,
    month,
    batch_size=100,
    scale=30
):

    region = rice_region_to_ee(
        rice_gdf
    )

    start = ee.Date.fromYMD(
        year,
        month,
        1
    )

    end = start.advance(
        1,
        "month"
    )

    s2 = get_s2(
        region,
        start,
        end
    )

    s1 = get_s1(
        region,
        start,
        end
    )

    if s2.size().getInfo() == 0:
        return pd.DataFrame()

    optical = (
        s2
        .select(
            [
                "NDVI",
                "EVI",
                "NDMI",
                "NDWI",
                "MNDWI"
            ]
        )
        .median()
    )

    bands = optical

    if s1.size().getInfo() > 0:

        sar = s1.median()

        vv_minus_vh = (
            sar.select("VV")
            .subtract(sar.select("VH"))
            .rename("VV_minus_VH")
        )

        bands = bands.addBands(
            sar.select(["VV", "VH"])
        ).addBands(
            vv_minus_vh
        )

    rows = []

    for i in range(
        0,
        len(rice_gdf),
        batch_size
    ):

        chunk = rice_gdf.iloc[
            i:i + batch_size
        ].copy()

        fc = gdf_chunk_to_ee(chunk)

        stats = bands.reduceRegions(
            collection=fc,
            reducer=ee.Reducer.mean(),
            scale=scale,
            tileScale=4
        )

        info = stats.getInfo()

        for f in info["features"]:

            props = f["properties"].copy()

            props["month"] = month

            rows.append(props)

    return pd.DataFrame(rows)

if RUN_POLYGON_LEVEL_SATELLITE:

    polygon_sat_tables = []

    for year in DETAIL_YEARS:

        if year not in rice:
            continue

        for month in GROWING_MONTHS:

            cache_csv = (
                SAT_DIR /
                f"sat_polygon_{year}_{month:02d}.csv"
            )

            if cache_csv.exists():

                t = pd.read_csv(cache_csv)

            else:

                t = polygon_month_satellite(
                    rice[year],
                    year,
                    month
                )

                t.to_csv(
                    cache_csv,
                    index=False
                )

            polygon_sat_tables.append(t)

    satellite_polygon_monthly = (
        pd.concat(
            polygon_sat_tables,
            ignore_index=True
        )
    )

    satellite_polygon_monthly.to_csv(
        OUT /
        "satellite_polygon_monthly.csv",
        index=False
    )

    display(
        satellite_polygon_monthly.head()
    )
else:
    print(
        "Polygon-level satellite is OFF. "
        "Set RUN_POLYGON_LEVEL_SATELLITE=True "
        "after region-level validation."
    )

# 6. Integration — Monthly Observation Table

For the lightning talk, first build a region-level table with:

```text
year
month
D2plus_pct
NDVI
EVI
NDMI
NDWI
MNDWI
SAR_VV
SAR_VH
production
yield
planted_acres
```

The next stage can refine this to:

```text
polygon_id × year × month
```


In [ ]:
analysis_monthly = (
    sat_region_monthly
    .merge(
        drought_region_monthly[
            [
                "year",
                "month",
                "d2plus_pct"
            ]
        ],
        on=["year", "month"],
        how="left"
    )
    .merge(
        nass_yearly[
            [
                "year",
                "planted_acres",
                "harvested_acres",
                "yield",
                "production"
            ]
        ],
        on="year",
        how="left"
    )
)

display(analysis_monthly)

analysis_monthly.to_csv(
    OUT /
    "analysis_monthly_macro_to_micro.csv",
    index=False
)

# 7. Lightning-Talk Output Checklist

## LT1 — Monitoring

Figures to complete:

1. **2000–latest NASS production time series**
2. **DWR rice-acreage maps for the detailed years**
3. **Monthly D2+ / D3+ drought exposure for the detailed years**
4. **Monthly NDVI / water / SAR signals for the detailed years**
5. If possible, a spatial map or animation

Story:

```text
Long-term change
      ↓
Where did it happen?
      ↓
Was drought present?
      ↓
What did optical / SAR observe?
```

## LT2 — Early-Supply Hypothesis

```text
April / May / June / July
drought + NDVI/EVI + NDWI + SAR + acreage
              ↓
Can these provide an early signal
before final production is known?
```

---

### Research Rule

Do not conclude that:

> “drought caused the production decline”

from correlation alone.

First observe the pattern, examine the spatial and temporal relationships, and then update the hypothesis.
